In [ ]:
import pandas as pd
from polotrial_client import PoloTrialClient

In [ ]:
def extrair_dado_dic(x):
    
    if isinstance(x, dict):
        if 'ds_descricao' in x:
            return x['ds_descricao']
        if 'descricao' in x:
            return x['descricao']
        if 'apelido_protocolo' in x:
            return x['apelido_protocolo']
        
        
    return None

In [ ]:

import pandas as pd
from polotrial_client import PoloTrialClient

def carregar_dados_protocolo(client: PoloTrialClient, protocolos: list[str]) -> pd.DataFrame:
    """
    Carrega os dados dos protocolos fornecidos usando o cliente Polotrial.

    Args:
        client (PoloTrialClient): Instância do cliente Polotrial.
        protocolos (list[str]): Lista de protocolos a serem carregados.

    Returns:
        pd.DataFrame: DataFrame contendo os dados dos protocolos.
    """
    protocolos_data = client.get_protocolos(protocolos)
    
    # Converte a lista de dicionários em um DataFrame
    df_protocolos = pd.DataFrame(protocolos_data)
    
    #Isolando as colunas de interesse para o dataframe
    df_protocolos = df_protocolos[[
        'id',
        'apelido_protocolo',
        'data_inicio_recrutamento',
        'data_fim_recrutamento',
        'status_protocolo',
        'dados_status_protocolo',
        'dados_patrocinador',
        'dados_co_centro'        
    ]]
    
    #Renomeando as colunas para facilitar a leitura
    df_protocolos.rename(columns={
        'id': 'id_protocolo',
        'apelido_protocolo': 'Protocolo',
        'data_inicio_recrutamento': 'Data de Início do Recrutamento',
        'data_fim_recrutamento': 'Data de Fim do Recrutamento',
        'status_protocolo': 'id_status_protocolo',
        'dados_status_protocolo': 'Status do Protocolo',
        'dados_patrocinador': 'Patrocinador',
        'dados_co_centro': 'Centro'
    }, inplace=True)
    
    #extrair dados dos dicionários dentro do daraframe
    colunas = ['Status do Protocolo', 'Patrocinador', 'Centro']
    for coluna in colunas:
        df_protocolos[coluna] = df_protocolos[coluna].apply(extrair_dado_dic)
    
    # Padronizar datas para o formato brasileiro
    df_protocolos['Data de Início do Recrutamento'] = pd.to_datetime(df_protocolos['Data de Início do Recrutamento']).dt.strftime('%d/%m/%Y')
    df_protocolos['Data de Fim do Recrutamento'] = pd.to_datetime(df_protocolos['Data de Fim do Recrutamento']).dt.strftime('%d/%m/%Y')
    
    return df_protocolos

if __name__ == "__main__":
    from dotenv import load_dotenv
    import os
    load_dotenv(override=True)
    BASE_URL = os.getenv("POLOTRIAL_API_URL")
    USERNAME = os.getenv("POLOTRIAL_API_USERNAME")
    PASSWORD = os.getenv("POLOTRIAL_API_PASSWORD")
    
    # Exemplo de uso
    client = PoloTrialClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
    protocolos = ["protocolo1", "protocolo2"]
    df_protocolos = carregar_dados_protocolo(client, protocolos)
    print(df_protocolos.head())

In [ ]:
#Armazenar os status únicos em uma lista para posterior uso, como filtragem ou análise.
lista_status = df_protocolos.copy()
lista_status = lista_status[['id_status_protocolo', 'Status do Protocolo']]
lista_status = lista_status.drop_duplicates().reset_index(drop=True)

In [ ]:

def carregar_dados_participante(client: PoloTrialClient, participantes: list[str]) -> pd.DataFrame:
    """
    Carrega os dados dos participantes fornecidos usando o cliente Polotrial.

    Args:
        client (PoloTrialClient): Instância do cliente Polotrial.
        participantes (list[str]): Lista de participantes a serem carregados.

    Returns:
        pd.DataFrame: DataFrame contendo os dados dos participantes.
    """
    participantes_data = client.get_participantes(participantes)
    
    # Converte a lista de dicionários em um DataFrame
    df_participantes = pd.DataFrame(participantes_data)
    
    df_participantes = df_participantes[[
        'id',
        'numero_de_screening',
        'co_protocolo',
        'dados_protocolo',
        'dados_status',
        'data_inclusao',
        'data_randomizacao'
    ]]
    
    df_participantes.rename(columns={
        
        'id': 'id_participante',
        'numero_de_screening': 'Número de Screening',
        'co_protocolo': 'id_protocolo',
        'dados_protocolo': 'Protocolo',
        'dados_status': 'Status do Participante',
        'data_inclusao': 'Data de Inclusão',
        'data_randomizacao': 'Data de Randomização'
    }, inplace=True)
    
    #Dados para extrair
    colunas = ['Protocolo', 'Status do Participante']
    for coluna in colunas:
        df_participantes[coluna] = df_participantes[coluna].apply(extrair_dado_dic)
    
    # Padronizar datas para o formato brasileiro
    df_participantes['Data de Inclusão'] = pd.to_datetime(df_participantes['Data de Inclusão']).dt.strftime('%d/%m/%Y')
    df_participantes['Data de Randomização'] = pd.to_datetime(df_participantes['Data de Randomização']).dt.strftime('%d/%m/%Y')
                            
    
    
    
    
    return df_participantes

if __name__ == "__main__":
    from dotenv import load_dotenv
    import os
    load_dotenv(override=True)
    BASE_URL = os.getenv("POLOTRIAL_API_URL")
    USERNAME = os.getenv("POLOTRIAL_API_USERNAME")
    PASSWORD = os.getenv("POLOTRIAL_API_PASSWORD")
    
    # Exemplo de uso
    client = PoloTrialClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
    participantes = ["participante1", "participante2"]
    df_participantes = carregar_dados_participante(client, participantes)
    print(df_participantes.head())

In [ ]:
protocolo_participante = df_participantes[[
    'id_participante',
    'Protocolo'
]]

In [ ]:
import pandas as pd
from polotrial_client import PoloTrialClient

def carregar_dados_participante_visita(client: PoloTrialClient, participante_visita: list[str]) -> pd.DataFrame:
    participante_visita_data = client.get_participante_visita(participante_visita)
    
    # Converte a lista de dicionários em um DataFrame
    df_participante_visita = pd.DataFrame(participante_visita_data)
    
    #Isolando as colunas de interesse para o dataframe
    df_participante_visita = df_participante_visita[[
        'id',
        'co_participante',
        'nome_tarefa',
        'data_estimada',
        'data_realizada',
        'valor_previsto',
        'dados_status',
        'dados_nota_fiscal'            
    ]]
    
    df_participante_visita.rename(columns={
        'id': 'id_participante_visita',
        'co_participante': 'id_participante',
        'nome_tarefa': 'Nome da Visita',
        'data_estimada': 'Data Estimada da Visita',
        'data_realizada': 'Data Realizada da Visita',
        'valor_previsto': 'Valor Previsto',
        'dados_status': 'Status da Visita',
        'dados_nota_fiscal': 'Nota Fiscal'
    }, inplace=True)
    
    #extrair dados dos dicionários dentro do daraframe
    def buscar_chave_recursiva(dic, chave_alvo):
        
        if not isinstance(dic, dict):
            return None
        if chave_alvo in dic:
            return dic[chave_alvo]
        for value in dic.values():
            if  isinstance(value, dict):
                resultado = buscar_chave_recursiva(value, chave_alvo)
                if resultado is not None:
                    return resultado
        return None
    def extrair_dado_generico(x):
        if isinstance(x, dict):
            codigo_nota_fiscal = buscar_chave_recursiva(x, 'codigo_nota_fiscal')
            descricao_nota_fiscal = buscar_chave_recursiva(x, 'ds_descricao')
            return codigo_nota_fiscal, descricao_nota_fiscal
        return None, None
    
    df_participante_visita['Codigo Nota Fiscal'], df_participante_visita['Descricao Nota Fiscal'] = zip(*df_participante_visita['Nota Fiscal'].apply(extrair_dado_generico))
    
    #Dados para extrair
    colunas = ['Status da Visita']
    for coluna in colunas:
        df_participante_visita[coluna] = df_participante_visita[coluna].apply(extrair_dado_dic)
        
    #Drop da coluna 'Nota Fiscal' após a extração dos dados
    df_participante_visita.drop(columns=['Nota Fiscal'], inplace=True)
    
        
    # Padronizar datas para o formato brasileiro
    df_participante_visita['Data Estimada da Visita'] = pd.to_datetime(df_participante_visita['Data Estimada da Visita']).dt.strftime('%d/%m/%Y')
    df_participante_visita['Data Realizada da Visita'] = pd.to_datetime(df_participante_visita['Data Realizada da Visita']).dt.strftime('%d/%m/%Y')
    
    #Data Merge com o dataframe protocolo_participante para adicionar a coluna 'Protocolo' ao dataframe df_participante_visita
    df_participante_visita = df_participante_visita.merge(protocolo_participante, on='id_participante', how='left')
    
    
    return df_participante_visita

if __name__ == "__main__":
    from dotenv import load_dotenv
    import os
    load_dotenv(override=True)
    BASE_URL = os.getenv("POLOTRIAL_API_URL")
    USERNAME = os.getenv("POLOTRIAL_API_USERNAME")
    PASSWORD = os.getenv("POLOTRIAL_API_PASSWORD")
    
    # Exemplo de uso
    client = PoloTrialClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
    participante_visita = ["participante_visita1", "participante_visita2"]
    df_participante_visita = carregar_dados_participante_visita(client, participante_visita)
    print(df_participante_visita.head())

In [ ]:
import pandas as pd
from polotrial_client import PoloTrialClient

def carregar_dados_participante_visita_procedimento_executor(client: PoloTrialClient) -> pd.DataFrame:
    participante_visita_procedimento_executor_data = client.get_participante_visita_procedimento_executor([])
    
    return pd.DataFrame(participante_visita_procedimento_executor_data)

if __name__ == "__main__":
    from dotenv import load_dotenv
    import os
    load_dotenv(override=True)
    BASE_URL = os.getenv("POLOTRIAL_API_URL")
    USERNAME = os.getenv("POLOTRIAL_API_USERNAME")
    PASSWORD = os.getenv("POLOTRIAL_API_PASSWORD")
    
    # Exemplo de uso
    client = PoloTrialClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)
    # participante_visita_procedimento_executor = ["executor1", "executor2"]
    df_participante_visita_procedimento_executor = carregar_dados_participante_visita_procedimento_executor(client)
    print(df_participante_visita_procedimento_executor.head())